# Embedding degli esempi di classificazione del router (versione_2)

Calcola gli embedding delle 112 domande etichettate in
`architetture_alternative/versione_2/esempi_classificazione.csv` — gli esempi
con cui il router (`architetture_alternative/versione_2/router.py`) confronta
per similarità coseno ogni nuova domanda in arrivo (vedi `classifica()`).

**Perché questo notebook, invece di calcolarli a runtime dentro `router.py`**:
stesso principio di `ingestion.ipynb` per film/biografie — la generazione degli
embedding è un lavoro di preparazione dati **una tantum**, non una responsabilità
del codice di servizio. Separandola qui, `router.py` diventa un consumatore
puramente offline degli esempi (legge un JSON già pronto, non chiama mai
l'API di embedding per loro) — la sola chiamata di embedding a runtime resta
quella per la domanda dell'utente in arrivo, che non può essere precalcolata.

Output: `architetture_alternative/versione_2/esempi_classificazione_embeddings.json`
— un dizionario `{domanda: embedding}`, letto da `router.py` all'avvio.

Come per `ingestion.ipynb`, il processo è **checkpointato**: rieseguibile senza
ricalcolare (e ripagare in quota) gli embedding già presenti nel file di
output, o — alla primissima esecuzione — in una vecchia cache locale del
router se presente (migrazione, vedi cella dedicata).

In [1]:
import json
import os
import time
import csv
from pathlib import Path

from dotenv import load_dotenv

# Radice del progetto: risale le cartelle a partire dalla working directory
# del kernel finche' non trova pyproject.toml — stesso approccio di ingestion.ipynb,
# necessario perche' questo notebook vive in ingestion/ ma i percorsi che usa
# (architetture_alternative/versione_2/..., .env) sono relativi alla radice.
def _find_project_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (parent / marker).exists():
            return parent
    raise RuntimeError(f"Impossibile trovare la radice del progetto (marker: {marker})")

PROJECT_ROOT = _find_project_root()
load_dotenv(dotenv_path=str(PROJECT_ROOT / ".env"))

V2_DIR = PROJECT_ROOT / "architetture_alternative" / "versione_2"
ESEMPI_CSV_PATH = V2_DIR / "esempi_classificazione.csv"
OUTPUT_EMBEDDINGS_PATH = V2_DIR / "esempi_classificazione_embeddings.json"
# Vecchia cache locale del router (pre-refactor): se presente, i suoi embedding
# vengono riusati come checkpoint iniziale invece di essere ricalcolati.
VECCHIA_CACHE_PATH = V2_DIR / "tmp" / "router_exemplars_cache.json"

print(f"Radice del progetto: {PROJECT_ROOT}")
print(f"CSV esempi: {ESEMPI_CSV_PATH}")
print(f"Output embedding: {OUTPUT_EMBEDDINGS_PATH}")

Radice del progetto: /Users/piermarone/Desktop/Progetto AgenticAI
CSV esempi: /Users/piermarone/Desktop/Progetto AgenticAI/architetture_alternative/versione_2/esempi_classificazione.csv
Output embedding: /Users/piermarone/Desktop/Progetto AgenticAI/architetture_alternative/versione_2/esempi_classificazione_embeddings.json


In [2]:
with ESEMPI_CSV_PATH.open(newline="", encoding="utf-8") as f:
    righe = list(csv.DictReader(f))

domande = [r["domanda"] for r in righe]
print(f"{len(domande)} domande caricate da {ESEMPI_CSV_PATH.name}")

from collections import Counter
print(Counter(r["categoria"] for r in righe))

112 domande caricate da esempi_classificazione.csv
Counter({'attore_film': 16, 'regista_film': 15, 'pitch_completo': 14, 'collaboratori_regista': 13, 'collaborazione_attori': 12, 'semantica_trame': 12, 'conteggio_genere': 11, 'semantica_biografia': 11, 'network_genere': 8})


## Checkpoint: carica gli embedding già calcolati

Prima il file di output definitivo (se questo notebook è già stato eseguito almeno una volta), poi — solo come migrazione una tantum — la vecchia cache del router, per non ripagare in quota Gemini gli embedding calcolati prima di questo refactor.

In [3]:
embeddings: dict[str, list[float]] = {}

if OUTPUT_EMBEDDINGS_PATH.exists():
    embeddings.update(json.loads(OUTPUT_EMBEDDINGS_PATH.read_text(encoding="utf-8")))
    print(f"Recuperati {len(embeddings)} embedding dal file di output esistente.")
elif VECCHIA_CACHE_PATH.exists():
    embeddings.update(json.loads(VECCHIA_CACHE_PATH.read_text(encoding="utf-8")))
    print(f"Migrati {len(embeddings)} embedding dalla vecchia cache del router ({VECCHIA_CACHE_PATH}).")
else:
    print("Nessun checkpoint trovato: si parte da zero.")

mancanti = [d for d in domande if d not in embeddings]
print(f"Da calcolare: {len(mancanti)} / {len(domande)}")

Migrati 112 embedding dalla vecchia cache del router (/Users/piermarone/Desktop/Progetto AgenticAI/architetture_alternative/versione_2/tmp/router_exemplars_cache.json).
Da calcolare: 0 / 112


## Calcolo degli embedding mancanti

Stesso modello e stessa configurazione usati da `router.py` per la domanda in arrivo — `gemini-embedding-001`, `task_type="SEMANTIC_SIMILARITY"` (confronto simmetrico tra due testi, non retrieval documentale), 768 dimensioni (ridotte rispetto alle 1536 del knowledge base principale, sufficienti per confrontare domande tra loro). Retry con backoff sui 429, come nel resto del progetto.

In [4]:
from agno.knowledge.embedder.google import GeminiEmbedder

embedder = GeminiEmbedder(
    id="gemini-embedding-001",
    api_key=os.getenv("GEMINI_API_KEY"),
    task_type="SEMANTIC_SIMILARITY",
    dimensions=768,
)

MAX_RETRIES = 3
BASE_RETRY_SLEEP = 20  # secondi, raddoppia a ogni tentativo

calcolati, errori = 0, 0
for i, domanda in enumerate(mancanti, 1):
    for tentativo in range(MAX_RETRIES):
        try:
            embeddings[domanda] = embedder.get_embedding(domanda)
            calcolati += 1
            break
        except Exception as exc:
            if "429" in str(exc) and tentativo < MAX_RETRIES - 1:
                attesa = BASE_RETRY_SLEEP * (2 ** tentativo)
                print(f"  [429] Attendo {attesa}s prima di riprovare...")
                time.sleep(attesa)
            else:
                print(f"  [Errore] '{domanda[:60]}': {exc}")
                errori += 1
                break
    if i % 20 == 0 or i == len(mancanti):
        print(f"  {i}/{len(mancanti)} — calcolati: {calcolati}, errori: {errori}")

print(f"\nCompletato: {calcolati} nuovi embedding calcolati, {errori} errori.")


Completato: 0 nuovi embedding calcolati, 0 errori.


## Salvataggio

In [5]:
OUTPUT_EMBEDDINGS_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_EMBEDDINGS_PATH.write_text(json.dumps(embeddings, ensure_ascii=False), encoding="utf-8")
print(f"Salvati {len(embeddings)} embedding in {OUTPUT_EMBEDDINGS_PATH}")

Salvati 112 embedding in /Users/piermarone/Desktop/Progetto AgenticAI/architetture_alternative/versione_2/esempi_classificazione_embeddings.json


## Verifica

In [6]:
mancanti_finali = [d for d in domande if d not in embeddings]
if mancanti_finali:
    print(f"⚠️  {len(mancanti_finali)} domande ancora senza embedding (probabile quota esaurita — rilancia il notebook più tardi):")
    for d in mancanti_finali:
        print("  -", d[:80])
else:
    print(f"✅ Tutte le {len(domande)} domande hanno un embedding. Il router può usare il file senza fare altre chiamate API per gli esempi.")

dimensioni = {len(v) for v in embeddings.values()}
print(f"Dimensionalità degli embedding presenti: {dimensioni}")

✅ Tutte le 112 domande hanno un embedding. Il router può usare il file senza fare altre chiamate API per gli esempi.
Dimensionalità degli embedding presenti: {768}
